In [1]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
import pandas as pd
from swmm_api import read_rpt_file, read_inp_file


# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path(
    r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427"
)

PROJECT = "ShawRd_pump"   # <-- change project name here
STORM = "Q3_50P"

storm_dir = ROOT / PROJECT / STORM

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_floodable_nodes(inp):
    """
    Return only junctions and storage units.

    Excludes:
    - Outfalls
    - Dividers
    """

    floodable_nodes = set()

    if inp.JUNCTIONS is not None:
        floodable_nodes.update(inp.JUNCTIONS.keys())

    if inp.STORAGE is not None:
        floodable_nodes.update(inp.STORAGE.keys())

    return floodable_nodes


def get_flooding_table(rpt, prefix, floodable_nodes):
    """
    Extract node flooding table from a SWMM report.

    Only keeps junctions and storage units.
    If no flooding occurred, returns an empty table.
    """

    cols = [
        f"{prefix} Hours Flooded",
        f"{prefix} Total Flood Vol (MG)"
    ]

    if rpt.node_flooding_summary is None:
        df = pd.DataFrame(columns=cols)
        df.index.name = "Node"
        return df

    df = rpt.node_flooding_summary.copy()
    df.index.name = "Node"

    # Keep only junctions and storage units
    df = df.loc[df.index.isin(floodable_nodes)]

    df = df.rename(columns={
        "Hours_Flooded": f"{prefix} Hours Flooded",
        "Total_Flood_Volume_10^6 gal": f"{prefix} Total Flood Vol (MG)",
    })

    return df[cols]


def summarize_simulation(pre_rpt, sim_rpt, sim_name, floodable_nodes):
    """
    Summarize one simulation using the pre-project flooded
    junctions/storage units as the fixed comparison population.

    Metrics:
    - Network total flood volume
    - Average node flood duration
    - Percent network flooding
    """

    # Extract pre-project and simulation flooding tables
    pre = get_flooding_table(pre_rpt, "Pre", floodable_nodes)
    sim = get_flooding_table(sim_rpt, "Sim", floodable_nodes)

    # Merge tables so nodes missing from either simulation become zero
    comparison = pre.join(sim, how="outer")

    fill_cols = [
        "Pre Hours Flooded",
        "Sim Hours Flooded",
        "Pre Total Flood Vol (MG)",
        "Sim Total Flood Vol (MG)"
    ]

    for col in fill_cols:
        comparison[col] = comparison[col].fillna(0)

    # Total floodable nodes = junctions + storage units
    total_nodes = len(floodable_nodes)

    # Fixed reference population = nodes that flooded pre-project
    pre_flooded_mask = comparison["Pre Hours Flooded"] > 0
    pre_flooded_nodes = pre_flooded_mask.sum()

    if sim_name == "Pre-Project Baseline":
        total_flood_volume_mg = comparison["Pre Total Flood Vol (MG)"].sum()

        avg_node_duration = comparison.loc[
            pre_flooded_mask,
            "Pre Hours Flooded"
        ].mean()

        flooded_nodes = pre_flooded_nodes

    else:
        total_flood_volume_mg = comparison["Sim Total Flood Vol (MG)"].sum()

        # Average duration across same nodes that flooded pre-project.
        # Nodes that no longer flood count as 0.
        avg_node_duration = comparison.loc[
            pre_flooded_mask,
            "Sim Hours Flooded"
        ].mean()

        # Count how many of the pre-project flooded nodes still flood.
        flooded_nodes = (
            comparison.loc[
                pre_flooded_mask,
                "Sim Hours Flooded"
            ] > 0
        ).sum()

    avg_node_duration = 0 if pd.isna(avg_node_duration) else avg_node_duration

    percent_network_flooded = (
        flooded_nodes / total_nodes * 100
        if total_nodes > 0 else 0
    )

    return {
        "Simulation": sim_name,
        "Network Total Flood Volume (MG)": round(total_flood_volume_mg, 3),
        "Avg Node Flood Duration (hrs)": round(avg_node_duration, 2),
        "Percent of Nodes Flooded in Network": f"{percent_network_flooded:.2f}%"
    }


# ============================================================
# FIND INPUT AND REPORT FILES
# ============================================================

if not storm_dir.exists():
    raise FileNotFoundError(f"Storm folder not found: {storm_dir}")

inp_files = sorted(storm_dir.glob("*.inp"))
pre_files = sorted(storm_dir.glob("*_pre.rpt"))
imp_files = sorted(storm_dir.glob("*_imp*.rpt"))

if not inp_files:
    raise FileNotFoundError(f"No .inp file found in: {storm_dir}")

if not pre_files:
    raise FileNotFoundError(f"No pre-project .rpt file found in: {storm_dir}")

inp_path = inp_files[0]
pre_path = pre_files[0]


# ============================================================
# LOAD MODEL AND PRE-PROJECT REPORT
# ============================================================

inp = read_inp_file(str(inp_path))
pre_rpt = read_rpt_file(str(pre_path))

floodable_nodes = get_floodable_nodes(inp)

print(f"INP file: {inp_path.name}")
print(f"Pre-project RPT file: {pre_path.name}")
print(f"Floodable nodes counted: {len(floodable_nodes)}")


# ============================================================
# BUILD LONG SUMMARY TABLE
# Rows = simulations
# Columns = metrics
# ============================================================

summary_rows = []

print(f"[RUN] Pre-Project Baseline: {pre_path.name}")

summary_rows.append(
    summarize_simulation(
        pre_rpt=pre_rpt,
        sim_rpt=pre_rpt,
        sim_name="Pre-Project Baseline",
        floodable_nodes=floodable_nodes
    )
)

for imp_path in imp_files:

    raw_imp_name = imp_path.stem.split("_")[-1].lower()
    imp_num = raw_imp_name.replace("imp", "")
    sim_name = f"Improvement {imp_num}"

    print(f"[RUN] {sim_name}: {imp_path.name}")

    imp_rpt = read_rpt_file(str(imp_path))

    summary_rows.append(
        summarize_simulation(
            pre_rpt=pre_rpt,
            sim_rpt=imp_rpt,
            sim_name=sim_name,
            floodable_nodes=floodable_nodes
        )
    )

summary_long_df = pd.DataFrame(summary_rows)

summary_long_df = summary_long_df[[
    "Simulation",
    "Network Total Flood Volume (MG)",
    "Avg Node Flood Duration (hrs)",
    "Percent of Nodes Flooded in Network"
]]


# ============================================================
# BUILD WIDE DISPLAY TABLE
# Rows = metrics
# Columns = simulations
# ============================================================

summary_df = summary_long_df.set_index("Simulation").T
summary_df.index.name = "Metric"


# ============================================================
# DISPLAY AND EXPORT
# ============================================================

display(summary_df)

INP file: SSF_SDMP_ShawRd_pump_imp1.inp
Pre-project RPT file: SSF_SDMP_ShawRd_pump_pre.rpt
Floodable nodes counted: 8
[RUN] Pre-Project Baseline: SSF_SDMP_ShawRd_pump_pre.rpt
[RUN] Improvement 1: SSF_SDMP_ShawRd_pump_imp1.rpt
[RUN] Improvement 2: SSF_SDMP_ShawRd_pump_imp2.rpt
[RUN] Improvement 3: SSF_SDMP_ShawRd_pump_imp3.rpt


Simulation,Pre-Project Baseline,Improvement 1,Improvement 2,Improvement 3
Metric,,,,
Network Total Flood Volume (MG),0.556,0.421,0.101,0.0
Avg Node Flood Duration (hrs),4.13,1.21,0.62,0.0
Percent of Nodes Flooded in Network,100.00%,50.00%,50.00%,0.00%
